In [ ]:
# ============================================================
# Project root & path handling
# ------------------------------------------------------------
# Default: working directory
# ============================================================

from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd

# Determine project root
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "..")).expanduser().resolve()

# Make the project's `src` package importable (e.g. src.lib.assay_priority)
sys.path.insert(0, str(PROJECT_ROOT))

# Convenience function for building repo-relative paths
def p(rel_path):
    """
    Build an absolute path from a path relative to the project root.
    """
    return PROJECT_ROOT / rel_path

print("PROJECT_ROOT set to:", PROJECT_ROOT)

In [ ]:
# ============================================================
# Create temporary and output directories 
# ============================================================

from src.lib.data_paths import get_data_paths

DATA_DIR = p("data")
_paths = get_data_paths(DATA_DIR)
_paths.make_output_dirs()

INPUT_MAVES_DIR = _paths.input_maves_dir
MAVE_DATA_DIR = _paths.mave_data_dir
SUPPLEMENTARY_DATA_DIR = _paths.supplementary_data_dir
MAVE_CALIBRATION_DIR = _paths.mave_calibration_dir
MAVE_CALIBRATION_ODDSPATH_DIR = _paths.mave_calibration_oddspath_dir
PREDICTOR_CALIBRATION_GENE_SPECIFIC_DIR = _paths.predictor_calibration_gene_specific_dir
RECLASSIFICATION_DIR = _paths.reclassification_dir

In [ ]:
import pandas as pd

sankey_OP = pd.read_csv(RECLASSIFICATION_DIR/"integrated_variant_effect_dataset_analysis.csv.gz")

In [ ]:
sankey_OP_2 = sankey_OP[sankey_OP['Gene'] != 'SFPQ']

In [ ]:
sankey_OP_2['VariantNotes_OP'] = np.where(
    sankey_OP_2['splice_var_amino'] == 'Yes',
    'splice_variant',
    ''
)

In [ ]:
start_lost = (
    (sankey_OP_2['nucleotide_or_aa'] == 'aa') &
    (sankey_OP_2['simplified_consequence'] == 'start_lost')
)

tag = 'start_lost_variant_not_measured'

existing = sankey_OP_2.loc[start_lost, 'VariantNotes_OP'].fillna('').astype(str)

sankey_OP_2.loc[start_lost, 'VariantNotes_OP'] = np.where(
    existing != "",
    existing + ';' + tag,
    tag
)

In [ ]:
OP_nuc = sankey_OP_2[sankey_OP_2['nucleotide_or_aa'] == 'nucleotide']

OP_aa = sankey_OP_2[sankey_OP_2['nucleotide_or_aa'] == 'aa']

In [ ]:
#mark any variants that get the opposite evidence between two assays

import numpy as np

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']

OP_nuc['Chrom'] = OP_nuc['Chrom'].astype(str)
OP_nuc['hg38_start'] = OP_nuc['hg38_start'].astype(str)
OP_nuc['ref_allele'] = OP_nuc['ref_allele'].astype(str)
OP_nuc['alt_allele'] = OP_nuc['alt_allele'].astype(str)
OP_nuc['Gene'] = OP_nuc['Gene'].astype(str)


def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()

#2018 clinvar conflicting

conflict_mask_OP_18 = OP_nuc.groupby(group_cols)['OP_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_OP_18 = conflict_mask_OP_18.fillna(False)

mask_2_18_OP = conflict_mask_OP_18 & OP_nuc['VariantNotes_OP'].notna() & (OP_nuc['VariantNotes_OP'] != "")

OP_nuc.loc[mask_2_18_OP, 'VariantNotes_OP'] = OP_nuc.loc[mask_2_18_OP, 'VariantNotes_OP'] + ';conflicting_fxn_data'

OP_nuc.loc[conflict_mask_OP_18 & ~mask_2_18_OP, 'VariantNotes_OP'] = 'conflicting_fxn_data'

def get_first_abs_max_idx(x):
    # Treat NaN as 0
    x_filled = x.fillna(0)

    # Compute the max absolute value
    abs_max = x_filled.abs().max()

    # Find the FIRST index where abs value equals abs_max
    return x_filled[x_filled.abs() == abs_max].index[0]

idx_max_18_OP = OP_nuc.groupby(group_cols)['OP_points'].apply(
    lambda x: get_first_abs_max_idx(x)
)

#restrict to rows where Fxn_use_variant is NA/empty
mask_na_18_OP = OP_nuc['VariantNotes_OP'].isna() | (OP_nuc['VariantNotes_OP'] == "")

OP_nuc.loc[idx_max_18_OP[idx_max_18_OP.isin(OP_nuc[mask_na_18_OP].index)], 'VariantNotes_OP'] = 'first_max_fxn_pts'

In [ ]:
OP_aa['Ref_seq_transcript_ID_stripped'] = OP_aa['RefSeq Transcript ID'].str.replace(r'\.\d+$', '', regex=True)

OP_aa['aa_pos'] = pd.to_numeric(OP_aa['aa_pos'], errors='coerce')
OP_aa['aa_ref'] = OP_aa['aa_ref'].astype(str)
OP_aa['aa_alt'] = OP_aa['aa_alt'].astype(str)
OP_aa['Gene'] = OP_aa['Gene'].astype(str)
OP_aa['Ref_seq_transcript_ID_stripped'] = OP_aa['Ref_seq_transcript_ID_stripped'].astype(str)

group_cols_aa = ['Gene', 'aa_ref', 'aa_pos', 'aa_alt','Ref_seq_transcript_ID_stripped']

def has_opposite_signs(x):
    x = x.dropna()
    non_zero = x[x != 0]
    return (non_zero > 0).any() and (non_zero < 0).any()


conflict_mask_aa_18_OP = OP_aa.groupby(group_cols_aa)['OP_points'] \
    .transform(lambda x: has_opposite_signs(x))

conflict_mask_aa_18_OP = conflict_mask_aa_18_OP.fillna(False)

mask_aa_18_OP = conflict_mask_aa_18_OP & OP_aa['VariantNotes_OP'].notna() & (OP_aa['VariantNotes_OP'] != "")

OP_aa.loc[mask_aa_18_OP, 'VariantNotes_OP'] = OP_aa.loc[mask_aa_18_OP, 'VariantNotes_OP'] + ';conflicting_fxn_data'

OP_aa.loc[conflict_mask_aa_18_OP & ~mask_aa_18_OP, 'VariantNotes_OP'] = 'conflicting_fxn_data'


idx_max_aa_18_OP = OP_aa.groupby(group_cols_aa)['OP_points'].apply(
    lambda x: x.fillna(0)[x.fillna(0).abs() == x.fillna(0).abs().max()].index
).explode()

# convert to Index
idx_max_aa_18_OP = pd.Index(idx_max_aa_18_OP)

mask_na_aa_18_OP = OP_aa['VariantNotes_OP'].isna() | (OP_aa['VariantNotes_OP'] == "")

OP_aa.loc[idx_max_aa_18_OP.intersection(OP_aa[mask_na_aa_18_OP].index), 'VariantNotes_OP'] = 'max_fxn_pts'

In [ ]:
OP_sankey_full = pd.concat([OP_nuc, OP_aa])

In [ ]:
#take out conflicting functional data and splice variants that are not measured 

dis = ['conflicting_fxn_data',
       'splice_variant_not_measured',
       'splice_variant_not_measured;conflicting_fxn_data','start_lost_variant_not_measured']

sankey_OP_18 = OP_sankey_full[
    ~OP_sankey_full['VariantNotes_OP'].isin(dis) &
    (OP_sankey_full['splice_var_amino'] != 'Yes')
]

In [ ]:
sankey_OP_18 = sankey_OP_18[sankey_OP_18['Flag'] != '*']

In [ ]:
controls_OP_18 = sankey_OP_18[sankey_OP_18['clnsig_group_18_25'].isin(['Benign','Benign/Likely benign','Likely benign','Pathogenic',
                                       'Pathogenic/Likely pathogenic','Likely pathogenic'])]

In [ ]:
import numpy as np

one_plus_stars = [
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
]

priority_genes = ['BRCA1', 'PTEN', 'MSH2', 'TP53']

#create new column with clinvar stars for genes where 2018 calibrations are needed, and if not then 2025 
controls_OP_18['clinvar_star_18_25'] = np.where(
    controls_OP_18['Gene'].isin(priority_genes),  
    controls_OP_18['clinvar_star_2018'], 
    controls_OP_18['clinvar_star_2025']
)

In [ ]:
#remove clinvar conflicts and splice variants 

controls_OP_18_x2 = controls_OP_18[(controls_OP_18['clinvar_conflict_flag_18_25'] != 'has clinvar conflict') & 
(controls_OP_18['splice_var_amino'] != 'Yes')]

In [ ]:
controls_nuc_OP_18 = controls_OP_18_x2[controls_OP_18_x2['nucleotide_or_aa'] == 'nucleotide']

controls_aa_OP_18 = controls_OP_18_x2[controls_OP_18_x2['nucleotide_or_aa'] == 'aa']

In [ ]:
controls_nuc_drop_OP_18 = (controls_nuc_OP_18
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
controls_nuc_drop_OP_18 = controls_nuc_drop_OP_18[controls_nuc_drop_OP_18['clinvar_star_18_25'].isin(one_plus_stars)]

In [ ]:
from src.lib.assay_priority import ASSAY_PRIORITY_LIST

assay_priority_list = ASSAY_PRIORITY_LIST
assay_priority_map = {name: i for i, name in enumerate(assay_priority_list)}

controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["Dataset"].map(assay_priority_map)

controls_aa_OP_18["assay_priority"] = controls_aa_OP_18["assay_priority"].fillna(9999)

In [ ]:
group_cols_aa_cln = ["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"]

one_plus_stars = {
    'criteria provided, single submitter',
    'criteria provided, multiple submitters, no conflicts',
    'reviewed by expert panel',
    'criteria provided, conflicting classifications'
}

zero_stars = {
    'no classification for the single variant',
    'no classification provided','no assertion criteria provided'
}


def summarize_clnstar(series):
    sigs = set(series.dropna())

    # All missing
    if len(sigs) == 0:
        return "Unseen"

    one_star = any(val in one_plus_stars for val in sigs)
    zero_star = any(val in zero_stars for val in sigs)

    if one_star and zero_star:
        return "has_clinvar_star_conflict"

    if one_star:
        return "one_plus_star"

    if zero_star:
        return "zero_star"

    raise ValueError(
        f"Unexpected ClinVar review_status values encountered: {sigs}"
    )

controls_aa_OP_18["clinvar_star_18_25_group"] = (
    controls_aa_OP_18
    .groupby(group_cols_aa_cln)["clinvar_star_18_25"]
    .transform(summarize_clnstar)
)

In [ ]:
controls_aa_OP_18 = controls_aa_OP_18[controls_aa_OP_18['clinvar_star_18_25_group'].isin(['one_plus_star'])]

In [ ]:
controls_aa_drop_OP_REVEL_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_REVEL_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


controls_aa_drop_OP_AM_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_AM_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


controls_aa_drop_OP_MP2_Pejaver_18 = (
    controls_aa_OP_18[
        (controls_aa_OP_18['VariantNotes_OP'] == 'max_fxn_pts')
        & (controls_aa_OP_18['GenomeWide_MP2_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

In [ ]:
controls_no_dup_REVEL_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_REVEL_Pejaver_18])
controls_no_dup_AM_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_AM_Pejaver_18])
controls_no_dup_MP2_OP_18_Pejaver = pd.concat([controls_nuc_drop_OP_18,controls_aa_drop_OP_MP2_Pejaver_18])

In [ ]:
controls_REVEL_18_OP_Pejaver = controls_no_dup_REVEL_OP_18_Pejaver[controls_no_dup_REVEL_OP_18_Pejaver['revel_train_amino'] != 'Yes']
controls_MP2_18_OP_Pejaver = controls_no_dup_MP2_OP_18_Pejaver[controls_no_dup_MP2_OP_18_Pejaver['mp2_train_amino'] != 'Yes']

In [ ]:
def catch_mis_2(df, group_cols, points_col='Fxn_points'):
    """
    Handle duplicates by keeping the row with the highest functional points.
    """
    group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']
    df_sorted = df.sort_values(by=points_col, ascending=False, na_position='last')
    cleaned = df_sorted.drop_duplicates(subset=group_cols, keep='first')
    return cleaned

In [ ]:
#REVEL

group_cols = ['Gene', 'Chrom', 'hg38_start', 'ref_allele', 'alt_allele']


controls_REVEL_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_REVEL_18_OP_Pejaver,
    group_cols, points_col='Fxn_points'
)

controls_MP2_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_MP2_18_OP_Pejaver,
    group_cols, points_col='Fxn_points'
)

controls_AM_18_OP_Pejaver_cleaned  = catch_mis_2(
    controls_no_dup_AM_OP_18_Pejaver,
    group_cols, points_col='Fxn_points'
)

In [ ]:
#VUS, check all on the nucleotide level 
VUS_18_OP = sankey_OP_18[sankey_OP_18['clinvar_18_25'].isin(['Uncertain significance'])]

In [ ]:
VUS_no_dup_18_OP = (
    VUS_18_OP
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
VUS_no_dup_REVEL_18_OP = VUS_no_dup_18_OP[VUS_no_dup_18_OP['revel_train_amino'] != 'Yes']

In [ ]:
VUS_no_dup_mut_18_OP = VUS_no_dup_18_OP[VUS_no_dup_18_OP['mp2_train_amino'] != 'Yes']

In [ ]:
VUS_no_dup_AM_18_OP = VUS_no_dup_18_OP

In [ ]:
# Unseen nucleotide variants, sankey_f is already filtered for splice variants not measured, conflicting functional data, and Flags removed, need to remove training variants where appropriate

Unseen = sankey_OP_18[(sankey_OP_18['clinvar_sig_2025'].isna()) & (sankey_OP_18['gnomad_MAF'].isna())]

In [ ]:
unseen_no_dup = (
    Unseen
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
#filter for SNVs
unseen_no_dup = unseen_no_dup[
    (unseen_no_dup['ref_allele'].str.len() == 1) &
    (unseen_no_dup['alt_allele'].str.len() == 1)
]

In [ ]:
unseen_no_dup_REVEL = unseen_no_dup[unseen_no_dup['revel_train_amino'] != 'Yes']

unseen_no_dup_mut = unseen_no_dup[unseen_no_dup['mp2_train_amino'] != 'Yes']

unseen_no_dup_AM = unseen_no_dup

In [ ]:
gnomad = sankey_OP_18[sankey_OP_18['gnomad_MAF'].notna()]

In [ ]:
gnomad_no_dup = (
    gnomad
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
gnomad_no_dup_REVEL = gnomad_no_dup[gnomad_no_dup['revel_train_amino'] != 'Yes']

gnomad_no_dup_mut = gnomad_no_dup[gnomad_no_dup['mp2_train_amino'] != 'Yes']

gnomad_no_dup_AM = gnomad_no_dup

In [ ]:
clingen = sankey_OP_18[sankey_OP_18['Updated_Classification_ClinGen_repo'].notna() & (sankey_OP_18['Updated_Classification_ClinGen_repo'] != 'VUS')]

In [ ]:
clingen_nuc = clingen[clingen['nucleotide_or_aa'] == 'nucleotide']

clingen_aa = clingen[clingen['nucleotide_or_aa'] == 'aa']

In [ ]:
clingen_nuc_drop = (clingen_nuc
    .sort_values(by="VariantNotes_OP", na_position="last") 
    .drop_duplicates(subset=['Gene', 'hg38_start', 'ref_allele', 'alt_allele'], keep="first")
)

In [ ]:
assay_priority_map = {name: i for i, name in enumerate(assay_priority_list)}

clingen_aa["assay_priority"] = clingen_aa["Dataset"].map(assay_priority_map)

clingen_aa["assay_priority"] = clingen_aa["assay_priority"].fillna(9999)

In [ ]:
clingen_aa_drop_REVEL_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_REVEL_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

clingen_aa_drop_mut_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_MP2_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)

clingen_aa_drop_AM_Pejaver = (clingen_aa[
        (clingen_aa['VariantNotes'] == 'max_fxn_pts')
        & (clingen_aa['GenomeWide_AM_max'] == 'max_pred_pts')
    ]
    .sort_values("assay_priority")
    .drop_duplicates(subset=["Gene", "aa_pos", "aa_ref", "aa_alt","Ref_seq_transcript_ID_stripped"], keep="first")
)


In [ ]:
clingen_no_dup_REVEL_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_REVEL_Pejaver])
clingen_no_dup_mut_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_mut_Pejaver])
clingen_no_dup_AM_Pejaver = pd.concat([clingen_nuc_drop,clingen_aa_drop_AM_Pejaver])

In [ ]:
ClinGen_repo_REVEL_Pejaver = clingen_no_dup_REVEL_Pejaver[clingen_no_dup_REVEL_Pejaver['revel_train_amino'] != 'Yes']
ClinGen_repo_mut_Pejaver = clingen_no_dup_mut_Pejaver[clingen_no_dup_mut_Pejaver['mp2_train_amino'] != 'Yes']
ClinGen_repo_AM_Pejaver = clingen_no_dup_AM_Pejaver

In [ ]:
#REVEL
ClinGen_repo_REVEL_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_REVEL_Pejaver,
    group_cols, points_col='Fxn_points'
)

ClinGen_repo_mut_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_mut_Pejaver,
    group_cols, points_col='Fxn_points'
)

ClinGen_repo_AM_Pejaver_cleaned  = catch_mis_2(
    ClinGen_repo_AM_Pejaver,
    group_cols, points_col='Fxn_points'
)

In [ ]:
dfs = {
    "controls_REVEL_OP": controls_REVEL_18_OP_Pejaver_cleaned,
    "controls_MP2_OP": controls_MP2_18_OP_Pejaver_cleaned,
    "controls_AM_OP": controls_AM_18_OP_Pejaver_cleaned,
    
    "VUS_REVEL_OP" : VUS_no_dup_REVEL_18_OP,
    "VUS_MP2_OP" : VUS_no_dup_mut_18_OP,
    "VUS_AM_OP" : VUS_no_dup_AM_18_OP,

    "ClinGen_Repo_REVEL_OP": ClinGen_repo_REVEL_Pejaver_cleaned,
    "ClinGen_repo_MP2_OP": ClinGen_repo_mut_Pejaver_cleaned,
    "ClinGen_repo_AM_OP": ClinGen_repo_AM_Pejaver_cleaned,
    
    "gnomAD_REVEL_OP": gnomad_no_dup_REVEL,
    "gnomAD_AM_OP": gnomad_no_dup_AM,
    "gnomAD_MP2_OP": gnomad_no_dup_mut,
    
    "Unobserved_REVEL_OP" : unseen_no_dup_REVEL,
    "Unobserved_AM_OP" : unseen_no_dup_AM,
    "Unobserved_mut_OP" : unseen_no_dup_mut
    
    
}


In [ ]:
COLUMNS_TO_DROP = ['REVEL_GenomeWide_Code', 'MP2_GenomeWide_Code', 'AM_GenomeWide_Code', 'REVEL_GeneSpecific_Code', 
                   'AM_GeneSpecific_Code', 'MP2_GeneSpecific_Code','Points_REVEL_GeneSpecific', 
                   'Points_AM_GeneSpecific', 'Points_MP2_GeneSpecific', 'Points_REVEL_GeneSpecific_GenomeWide', 
                   'Points_AM_GeneSpecific_GenomeWide', 'Points_MP2_GeneSpecific_GenomeWide', 'Total_Points_GenomeWide_REVEL', 
                   'Total_Points_GenomeWide_AM', 'Total_Points_GenomeWide_MP2', 'Total_Points_GeneSpecific_REVEL',
                   'Total_Points_GeneSpecific_AM', 'Total_Points_GeneSpecific_MP2', 'Class_GenomeWide_REVEL', 'Class_GenomeWide_AM', 
                   'Class_GenomeWide_MP2', 'Class_GeneSpecific_REVEL', 'Class_GeneSpecific_AM', 'Class_GeneSpecific_MP2', 
                   'Conflicting_REVEL_GenomeWide', 'Conflicting_AM_GenomeWide', 'Conflicting_MP2_GenomeWide', 
                   'Conflicting_REVEL_GeneSpecific', 'Conflicting_AM_GeneSpecific', 'Conflicting_MP2_GeneSpecific','splice_variant', 
                   'VariantNotes', 'GenomeWide_REVEL_max', 'GeneSpecific_REVEL_max', 'GenomeWide_AM_max', 'GeneSpecific_AM_max', 
                   'GenomeWide_MP2_max', 'GeneSpecific_MP2_max','clnsig_group_25','revel_train_amino', 'mp2_train_amino',
                   'splice_var_amino', 'clinvar_conflict_flag_18_25', 'VariantNotes_OP', 'Ref_seq_transcript_ID_stripped', 
                   'clinvar_star_18_25', 'assay_priority', 'clinvar_star_18_25_group'
    
]

In [ ]:
for name, df in dfs.items():
    dfs[name] = df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [ ]:
# Define new column names
rename_dict = {
    'Total_Points_OP_GenomeWide_REVEL': 'Total_Points_OP_REVEL',
    'Total_Points_OP_GenomeWide_AM': 'Total_Points_OP_AM',
    'Total_Points_OP_GenomeWide_MP2': 'Total_Points_OP_MP2',
    'ClassOP_GenomeWide_REVEL': 'Class_OP_REVEL', 
    'ClassOP_GenomeWide_AM':'Class_OP_AM',
    'ClassOP_GenomeWide_MP2':'Class_OP_MP2',
    'Conflicting_OP_REVEL_GenomeWide': 'Conflicting_OP_REVEL',
    'Conflicting_OP_AM_GenomeWide': 'Conflicting_OP_AM',
    'Conflicting_OP_MP2_GenomeWide': 'Conflicting_OP_MP2',
    'clinvar_18_25': 'clinvar_sig_18_25'

}

# Define column order
column_order = ['mavedb_variant_urn', 'Dataset', 'Gene', 'HGNC_id', 'Chrom', 'Strand', 'hg38_start', 'hg38_end', 'ref_allele', 'alt_allele', 
                'auth_transcript_id', 'transcript_pos', 'transcript_ref', 'transcript_alt', 'aa_pos', 'aa_ref', 'aa_alt', 'hgvs_c', 
                'hgvs_p', 'consequence', 'simplified_consequence', 'auth_reported_score',
                'auth_reported_func_class', 'splice_measure', 'gnomad_MAF', 'clinvar_sig_2025', 'clinvar_star_2025', 
                'clinvar_date_last_reviewed_2025', 'clinvar_sig_2018', 'clinvar_star_2018', 'clinvar_date_last_reviewed_2018', 
                'nucleotide_or_aa', 'Ensembl Transcript ID', 'RefSeq Transcript ID', 'Interval 1 Name', 'Interval 1 Range',
                'Interval 1 Class', 'Interval 2 Name', 'Interval 2 Range', 'Interval 2 Class', 'Interval 3 Name', 'Interval 3 Range', 
                'Interval 3 Class', 'Interval 4 Name', 'Interval 4 Range', 'Interval 4 Class', 'Interval 5 Name', 'Interval 5 Range', 
                'Interval 5 Class', 'Interval 6 Name', 'Interval 6 Range', 'Interval 6 Class', 'Flag', 'REVEL', 'REVEL_train', 
                'AM_score', 'AM_class', 'MutPred2', 'MP2_train', 'spliceAI_DS_AG', 'spliceAI_DS_AL', 'spliceAI_DS_DG', 'spliceAI_DS_DL',
                'spliceAI_DP_AG', 'spliceAI_DP_AL', 'spliceAI_DP_DG', 'spliceAI_DP_DL', 'ClinVar Variation Id_ClinGen_repo', 
                'Allele Registry Id_ClinGen_repo', 'Disease_ClinGen_repo', 'Mondo Id_ClinGen_repo', 'Mode of Inheritance_ClinGen_repo', 
                'Assertion_ClinGen_repo', 'Applied Evidence Codes (Met)_ClinGen_repo', 'Applied Evidence Codes (Not Met)_ClinGen_repo', 
                'Summary of interpretation_ClinGen_repo', 'PubMed Articles_ClinGen_repo', 'Expert Panel_ClinGen_repo', 
                'Guideline_ClinGen_repo', 'Approval Date_ClinGen_repo', 'Published Date_ClinGen_repo', 'Retracted_ClinGen_repo', 
                'Evidence Repo Link_ClinGen_repo', 'Uuid_ClinGen_repo', 'Updated_Classification_ClinGen_repo', 
                'Updated_Evidence Codes_ClinGen_repo','clinvar_sig_18_25','clnsig_group_18_25','StandardizedClass','ExC_points_2025', 
                'ExC_points_2018', 'OddsNormal', 'OddsAbnormal', 'OP_points', 'Fxn_points', 'Points_REVEL_GenomeWide', 
                'Points_AM_GenomeWide', 'Points_MP2_GenomeWide', 'Total_Points_OP_REVEL',
                'Total_Points_OP_AM', 'Total_Points_OP_MP2', 'Class_OP_REVEL', 
                'Class_OP_AM', 'Class_OP_MP2', 'Conflicting_OP_REVEL', 'Conflicting_OP_AM',
                'Conflicting_OP_MP2']

# Apply to all dataframes in dictionary
for key in dfs:
    dfs[key] = dfs[key].rename(columns=rename_dict)[column_order]

In [ ]:
out_folder = MAVE_CALIBRATION_ODDSPATH_DIR

for name, df in dfs.items():
    df.to_csv(out_folder / f"{name}.csv", index=False)

In [ ]:
import pandas as pd
import gzip
import shutil
from pathlib import Path

folder = MAVE_CALIBRATION_ODDSPATH_DIR
output = SUPPLEMENTARY_DATA_DIR / "Supplementary_Data_6.xlsx"

with pd.ExcelWriter(output) as writer:
    for csv_file in folder.glob("*.csv"):
        df = pd.read_csv(csv_file)
        sheet_name = csv_file.stem[:31]
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# Gzip the Excel file
with open(output, 'rb') as f_in:
    with gzip.open(str(output) + '.gz', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)